In [7]:
%run "D:\22\AA\AA journal\code\student\stat_clinical_significance\Kd_common.py"

In [9]:
import os

path = r"D:\22\AA\AA journal\code\student\stat_clinical_significance\Kd_common.py"

# আগে দেখি ফাইলটা সত্যিই কবে edit হয়েছে সবশেষ
print("Size before:", os.path.getsize(path))

function_code = '''

def save_per_seed_predictions_csv(out_csv, subject_ids, per_seed_preds, true_labels):
    """
    Long-format CSV: one row per (seed, epoch).
    """
    import csv as csv_module
    with open(out_csv, 'w', newline='') as f:
        writer = csv_module.writer(f)
        writer.writerow(["seed", "subject_id", "true_label", "predicted_label"])
        for seed, preds in per_seed_preds.items():
            for sid, t, p in zip(subject_ids, true_labels, preds):
                writer.writerow([seed, sid, int(t), int(p)])
'''

with open(path, "a", encoding="utf-8") as f:
    f.write(function_code)

print("Size after:", os.path.getsize(path))

# এখন verify করি
with open(path, "r", encoding="utf-8") as f:
    content = f.read()
print("Has function now:", "def save_per_seed_predictions_csv" in content)

Size before: 10360
Size after: 10898
Has function now: True


In [11]:
import importlib.util

path = r"D:\22\AA\AA journal\code\student\stat_clinical_significance\Kd_common.py"

# ১. ফাইলের raw content-এ function-টা আছে কিনা সরাসরি check করো
with open(path, "r", encoding="utf-8") as f:
    content = f.read()

print("File length:", len(content))
print("Has save_per_seed_predictions_csv:", "def save_per_seed_predictions_csv" in content)

# ২. Python আসলে কোন ফাইলটা import করছে সেটা confirm করো
spec = importlib.util.find_spec("Kd_common")
print("Python will import from:", spec.origin if spec else "NOT FOUND")

File length: 11714
Has save_per_seed_predictions_csv: True
Python will import from: D:\22\AA\AA journal\code\student\stat_clinical_significance\Kd_common.py


In [10]:
# ============================================================
# S1 -- WEARABLE-ONLY BASELINE: 5-seed soft-vote ensemble
# Produces ensemble_predictions.csv (with subject_id) for
# stat_clinical_significance.py
# ============================================================
# ============================================================
import os
import sys
sys.modules.pop('Kd_common', None)
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score

COMMON_PATH = r"D:\22\AA\AA journal\code\student\stat_clinical_significance"

if COMMON_PATH not in sys.path:
    sys.path.insert(0, COMMON_PATH)

from Kd_common import (
    load_3way_split,
    WearableBaselineModel,
    LABEL_NAMES,
    save_ensemble_predictions_csv,
    save_per_seed_predictions_csv
)
STUDENT_DATA_PATH = r"D:\22\AA\preprocess\preprocessed_student_zmax_e4"
SPLIT_PATH        = r"D:\22\AA\AA journal\preprocess\preprocessed_split_v2"
CKPT_DIR          = r"D:\22\AA\AA journal\evaluation\teacher-student\s1_wearable_baseline_valfixed"
OUT_DIR           = r"D:\22\AA\AA journal\evaluation\teacher-student\ensemble_s1_wearable_baseline"
os.makedirs(OUT_DIR, exist_ok=True)

SEEDS = [42, 123, 256, 789, 999]
D_MODEL = 96
DROPOUT = 0.3
BATCH_SIZE = 64

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


class SubjectTrackedWearableDataset(Dataset):
    """Same filtering/ordering as S1's WearableDataset, but also
    exposes subject_id per sample (parallel array, index-aligned)."""
    def __init__(self, subject_list, data_path):
        self.data = []
        self.index = []
        self.subject_ids = []
        for sub in subject_list:
            fp = os.path.join(data_path, f"{sub}.npz")
            if not os.path.exists(fp):
                continue
            with np.load(fp) as d:
                zmax_arr = d['zmax_eeg']
                e4_arr = d['e4']
                labels = d['labels'].copy()
            n = min(zmax_arr.shape[0], e4_arr.shape[0], len(labels))
            sub_idx = len(self.data)
            self.data.append((zmax_arr[:n], e4_arr[:n], labels[:n]))
            for i in range(n):
                self.index.append((sub_idx, i))
                self.subject_ids.append(sub)
        print(f"  Subjects: {len(self.data)}   Samples: {len(self.index):,}")

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        sub_idx, i = self.index[idx]
        zmax_arr, e4_arr, labels = self.data[sub_idx]
        return (torch.FloatTensor(zmax_arr[i]), torch.FloatTensor(e4_arr[i]),
                torch.tensor(int(labels[i]), dtype=torch.long))


_, _, TEST_SUBS = load_3way_split(SPLIT_PATH)
print(f"Test subjects (from split file): {len(TEST_SUBS)}")

print("Building test dataset...")
test_ds = SubjectTrackedWearableDataset(TEST_SUBS, STUDENT_DATA_PATH)
if len(test_ds) == 0:
    raise RuntimeError("test_ds empty -- check STUDENT_DATA_PATH.")

# shuffle=False is REQUIRED -- subject_ids array must line up with iteration order
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

models = []
loaded_seeds = []
print("\nLoading checkpoints...")
for seed in SEEDS:
    ckpt_path = os.path.join(CKPT_DIR, f"best_seed{seed}.pt")
    if not os.path.exists(ckpt_path):
        print(f"  WARNING: missing {ckpt_path}")
        continue
    m = WearableBaselineModel(d_model=D_MODEL, dropout=DROPOUT).to(device)
    m.load_state_dict(torch.load(ckpt_path, map_location=device))
    m.eval()
    for p in m.parameters():
        p.requires_grad = False
    models.append(m)
    loaded_seeds.append(seed)
    print(f"  Loaded seed {seed}")

if not models:
    raise RuntimeError("No checkpoints found in CKPT_DIR.")
print(f"Total models loaded: {len(models)}")

all_true, all_pred, all_probs = [], [], []
per_seed_preds = {seed: [] for seed in loaded_seeds}
with torch.no_grad():
    for zmax_x, e4_x, y in test_loader:
        zmax_x, e4_x = zmax_x.to(device), e4_x.to(device)
        prob_sum = None
        for seed, m in zip(loaded_seeds, models):
            logits = m(zmax_x, e4_x)
            probs = F.softmax(logits, dim=1)
            per_seed_preds[seed].extend(probs.argmax(dim=1).cpu().numpy())
            prob_sum = probs if prob_sum is None else prob_sum + probs
        avg_probs = (prob_sum / len(models)).cpu().numpy()
        preds = avg_probs.argmax(axis=1)
        all_true.extend(y.numpy())
        all_pred.extend(preds)
        all_probs.extend(avg_probs)

all_true = np.array(all_true)
all_pred = np.array(all_pred)
all_probs = np.array(all_probs)

acc = accuracy_score(all_true, all_pred)
f1 = f1_score(all_true, all_pred, average='macro', labels=[0, 1, 2, 3, 4], zero_division=0)
kappa = cohen_kappa_score(all_true, all_pred, labels=[0, 1, 2, 3, 4])
per_cls = f1_score(all_true, all_pred, average=None, labels=[0, 1, 2, 3, 4], zero_division=0)

print("\n" + "=" * 60)
print(f"S1 ENSEMBLE ({len(models)} seeds)")
print("=" * 60)
print(f"Accuracy : {acc*100:.2f}%")
print(f"Macro F1 : {f1:.4f}")
print(f"Kappa    : {kappa:.4f}")
for name, score in zip(LABEL_NAMES, per_cls):
    print(f"  {name:6s}: {score:.4f}")

out_csv = os.path.join(OUT_DIR, "ensemble_predictions.csv")
save_ensemble_predictions_csv(out_csv, test_ds.subject_ids, all_true, all_pred, all_probs)
print(f"\nSaved: {out_csv}")

per_seed_csv = os.path.join(OUT_DIR, "per_seed_predictions.csv")
save_per_seed_predictions_csv(per_seed_csv, test_ds.subject_ids, per_seed_preds, all_true)
print(f"Saved: {per_seed_csv}")
print("Done.")

Device: cuda
Test subjects (from split file): 20
Building test dataset...
  Subjects: 19   Samples: 18,899

Loading checkpoints...
  Loaded seed 42
  Loaded seed 123
  Loaded seed 256
  Loaded seed 789
  Loaded seed 999
Total models loaded: 5

S1 ENSEMBLE (5 seeds)
Accuracy : 56.06%
Macro F1 : 0.4954
Kappa    : 0.4053
  Wake  : 0.4607
  N1    : 0.2571
  N2    : 0.6443
  N3    : 0.5925
  REM   : 0.5226

Saved: D:\22\AA\AA journal\evaluation\teacher-student\ensemble_s1_wearable_baseline\ensemble_predictions.csv
Saved: D:\22\AA\AA journal\evaluation\teacher-student\ensemble_s1_wearable_baseline\per_seed_predictions.csv
Done.


In [8]:
with open(path, "r", encoding="utf-8") as f:
    content = f.read()
print("Has save_per_seed_predictions_csv:", "def save_per_seed_predictions_csv" in content)

Has save_per_seed_predictions_csv: True
